# Visualizations  

> This module is dedicated to visualizing single-cell RNA sequencing data, enhancing interpretability and insights across diverse datasets. It offers essential tools for data representation, exploration, and presentation, enabling effective analysis within the Allos framework.


In [ ]:
#| default_exp visuals

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
import pandas as pd
import anndata as ad

In [ ]:
#| export
import scanpy as sc

def plot_transcripts(adata, gene_id=None, transcripts=None):
    """
    Plot the UMAP with the specified transcripts or transcripts associated with the given gene ID.
    
    Parameters:
    -----------
    adata : AnnData
        The annotated data matrix.
    gene_id : str, optional
        The gene ID for which to find and plot associated transcripts. 
        Ignored if `transcripts` is provided.
    transcripts : list of str, optional
        A list of transcript IDs to plot. If provided, `gene_id` is ignored.
    
    Returns:
    --------
    None
        Displays the UMAP plot colored by the specified transcripts.
    """
    if transcripts is None:
        if gene_id is None:
            raise ValueError("Either `gene_id` or `transcripts` must be provided.")
        # Select transcripts associated with the provided gene ID
        transcripts = adata[:, adata.var['geneId'] == gene_id].var.index.to_list()
    
    # Plot UMAP
    sc.pl.umap(adata, color=transcripts)


In [ ]:
#| export
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm
import matplotlib.pyplot as plt

###############################################################################
# 1) Helper: approximate "hpi" bandwidth (1D) 
###############################################################################

def normal_reference_bandwidth(x):
    """
    Approximate the normal-reference (Silverman's) bandwidth for 1D data.
    R's ks::hpi uses a more sophisticated pilot estimation, but this is
    often 'close enough' for large data.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    sigma = np.std(x, ddof=1)
    bw = 0.9 * min(sigma, iqr / 1.34) * n ** (-1 / 5)
    return bw


###############################################################################
# 2) The core wkde2d function 
###############################################################################

def wkde2d(x, y, w=None, h=None, adjust=1.0, n=100, lims=None):
    """
    Python equivalent of R 'wkde2d' function:
      x, y: coordinates (1D arrays) for each observation.
      w   : weight vector (same length as x, y).
      h   : tuple or scalar for bandwidth in the x/y directions (if None, use approximate).
      adjust: bandwidth adjustment scalar.
      n   : number of grid points in each direction.
      lims: [x_min, x_max, y_min, y_max] to define the grid.
    Returns a dict with {'x': gx, 'y': gy, 'z': Z}.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if w is None:
        w = np.ones_like(x)
    else:
        w = np.asarray(w, dtype=float)

    if len(x) != len(y) or len(x) != len(w):
        raise ValueError("x, y, w must all have the same length")

    if lims is None:
        lims = [x.min(), x.max(), y.min(), y.max()]
    if len(lims) != 4:
        raise ValueError("lims must be [x_min, x_max, y_min, y_max]")

    if h is None:
        hx = normal_reference_bandwidth(x)
        hy = normal_reference_bandwidth(y)
        h = (hx, hy)
    else:
        if np.isscalar(h):
            h = (float(h), float(h))
        else:
            h = tuple(h)
    h = (h[0] * adjust, h[1] * adjust)

    gx = np.linspace(lims[0], lims[1], n)
    gy = np.linspace(lims[2], lims[3], n)

    ax = (gx[:, None] - x[None, :]) / h[0]
    ay = (gy[:, None] - y[None, :]) / h[1]

    fx = np.exp(-0.5 * ax**2) / np.sqrt(2.0 * np.pi)
    fy = np.exp(-0.5 * ay**2) / np.sqrt(2.0 * np.pi)

    w = w[None, :]
    fxw = fx * w
    fyw = fy * w

    z = fxw.dot(fyw.T)
    Z = z / (w.sum() * h[0] * h[1])

    return {"x": gx, "y": gy, "z": Z}


###############################################################################
# 3) Mapping per-cell densities
###############################################################################

def get_dens(points, dens):
    """
    Map each 2D point to the approximate density in `dens["z"]`.
       points: shape (n_cells, 2)
       dens  : dict with 'x', 'y', 'z', from wkde2d
    Returns a 1D np.array of densities (one per row in points).
    """
    xgrid = dens["x"]
    ygrid = dens["y"]
    Z     = dens["z"]

    ix = np.searchsorted(xgrid, points[:, 0]) - 1
    iy = np.searchsorted(ygrid, points[:, 1]) - 1

    ix = np.clip(ix, 0, len(xgrid) - 1)
    iy = np.clip(iy, 0, len(ygrid) - 1)

    return Z[ix, iy]


###############################################################################
# 4) Calculate density from AnnData
###############################################################################

def calculate_density(adata, feature, basis="umap", adjust=1.0, map_to_cells=True):
    """
    1) Extract coordinates from adata.obsm[f"X_{basis}"].
    2) Extract expression vector for `feature`.
    3) Run the weighted KDE.
    4) If map_to_cells=True, return per-cell densities; else return the full grid.
    """
    if f"X_{basis}" not in adata.obsm:
        raise ValueError(f"AnnData has no .obsm['X_{basis}']")

    coords = adata.obsm[f"X_{basis}"]
    if coords.shape[1] != 2:
        raise ValueError(f"Embedding {basis} must be 2D, found shape {coords.shape}")

    if feature in adata.var_names:
        w = adata.obs_vector(feature)
    elif feature in adata.obs.columns:
        w = adata.obs[feature].values
    else:
        raise ValueError(f"Feature '{feature}' not found in adata.var_names or adata.obs.columns")

    dens = wkde2d(
        x=coords[:, 0],
        y=coords[:, 1],
        w=w,
        adjust=adjust,
        n=200,
        lims=None,
    )

    if map_to_cells:
        return get_dens(coords, dens)
    else:
        return dens

In [ ]:
#| hide
from nbdev.showdoc import *